In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [ ]:
!pip install transformers

In [ ]:
!pip install --upgrade transformers tensorflow

In [ ]:
df_train = pd.read_csv('/content/train_cleaned_checkpoint.csv')
df_test = pd.read_csv('/content/test_cleaned_checkpoint.csv')
df_valid = pd.read_csv('/content/valid_cleaned_checkpoint.csv')

# **Label Encoding**


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# fit_transform
y_train = label_encoder.fit_transform(df_train['label'])

# transform
y_valid = label_encoder.transform(df_valid['label'])
y_test  = label_encoder.transform(df_test['label'])

# Class Mapping
class_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
print("Emotion Class Mapping:", class_mapping)

Emotion Class Mapping: {'anger': 0, 'fear': 1, 'happy': 2, 'love': 3, 'sadness': 4}


# **Training Model (IndoBERT & IndoBERTweet)**

In [ ]:
import torch
from torch.utils.data import Dataset

from transformers import AutoTokenizer
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import classification_report

In [ ]:
# Format PyTorch Dataset
class EmotDataset(Dataset):
    def __init__(self, encodings_ids, encodings_masks, labels):
        self.ids = encodings_ids
        self.masks = encodings_masks
        self.labels = labels

    def __getitem__(self, idx):
        return {
            # Memastikan tipe data long (integer) untuk PyTorch
            'input_ids': torch.tensor(self.ids[idx], dtype=torch.long),
            'attention_mask': torch.tensor(self.masks[idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

    def __len__(self):
        return len(self.labels)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## **A. IndoBERT**

### **1) Tokenization**

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "indolem/indobert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Clean tag [username] & [url]
def clean_bert(text):
    text = str(text).lower()
    text = re.sub(r'\[username\]|\[url\]', '', text)
    return text.strip()

text_train_bert = df_train['tweet'].apply(clean_bert).tolist()
text_valid_bert = df_valid['tweet'].apply(clean_bert).tolist()
text_test_bert  = df_test['tweet'].apply(clean_bert).tolist()

def prepare_bert_input(texts_list, max_len=43):
    encoded = bert_tokenizer(
        texts_list,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    return encoded['input_ids'], encoded['attention_mask']

# Tokenization
X_train_bert_ids, X_train_bert_mask = prepare_bert_input(text_train_bert)
X_valid_bert_ids, X_valid_bert_mask = prepare_bert_input(text_valid_bert)
X_test_bert_ids, X_test_bert_mask   = prepare_bert_input(text_test_bert)

train_dataset_bert = EmotDataset(X_train_bert_ids, X_train_bert_mask, y_train)
valid_dataset_bert = EmotDataset(X_valid_bert_ids, X_valid_bert_mask, y_valid)
test_dataset_bert  = EmotDataset(X_test_bert_ids,  X_test_bert_mask,  y_test)

print(f"X_train_bert_ids Dimension: {X_train_bert_ids.shape}")

X_train_bert_ids Dimension: torch.Size([3521, 43])


### **2) Custom Head**

In [ ]:
import torch.nn as nn
from transformers import AutoModel

class IndoBERTBaseCustom(nn.Module):
    def __init__(self, num_labels, class_weights=None):
        super(IndoBERTBaseCustom, self).__init__()
        self.num_labels = num_labels
        self.class_weights = class_weights

        self.bert = AutoModel.from_pretrained("indolem/indobert-base-uncased")

        # Arsitektur Klasifikasi Custom (Shaw et al.)
        self.dense1 = nn.Linear(self.bert.config.hidden_size, 64)
        self.dropout = nn.Dropout(0.05)
        self.dense2 = nn.Linear(64, 16)
        self.classifier = nn.Linear(16, num_labels)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # GLOBAL AVERAGE POOLING
        mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
        sum_embeddings = torch.sum(outputs.last_hidden_state * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask

        # CUSTOM CLASSIFICATION HEAD
        x = self.dense1(pooled_output)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.dense2(x)
        x = torch.relu(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            return {"loss": loss, "logits": logits}

        return {"logits": logits}

model_bert_custom = IndoBERTBaseCustom(
    num_labels=len(class_mapping)
).to(device)

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### **3) Training**

In [ ]:
# Hyperparameter (Shaw et al.)
training_args = TrainingArguments(
    output_dir='./indobert_base_results',
    num_train_epochs=25,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    weight_decay=0.005,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model_bert_custom,
    args=training_args,
    train_dataset=train_dataset_bert,
    eval_dataset=valid_dataset_bert,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

trainer.train()

print("\n[Evaluation IndoBERT on TEST Data]")
predictions = trainer.predict(test_dataset_bert)
y_pred_bert = np.argmax(predictions.predictions, axis=1)

print(classification_report(y_test, y_pred_bert, target_names=label_encoder.classes_))


--- Memulai Proses Fine-Tuning IndoBERT Base (Custom Head) ---


/tmp/ipykernel_3747/3758758826.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(self.ids[idx], dtype=torch.long),
/tmp/ipykernel_3747/3758758826.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'attention_mask': torch.tensor(self.masks[idx], dtype=torch.long),


Epoch,Training Loss,Validation Loss
1,1.529382,1.383682
2,1.238109,1.105637
3,0.933184,1.022763
4,0.743366,1.004062
5,0.591403,1.101015
6,0.495144,1.121049
7,0.386989,1.178913
8,0.308287,1.157548
9,0.252365,1.186586


/tmp/ipykernel_3747/3758758826.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(self.ids[idx], dtype=torch.long),
/tmp/ipykernel_3747/3758758826.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'attention_mask': torch.tensor(self.masks[idx], dtype=torch.long),
/tmp/ipykernel_3747/3758758826.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(self.ids[idx], dtype=torch.long),
/tmp/ipykernel_3747/3758758826.py:12: UserWarning: To copy construct from a tensor, it is recommended to 


[Evaluasi IndoBERT Base (Custom Head) pada Data TEST]


/tmp/ipykernel_3747/3758758826.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(self.ids[idx], dtype=torch.long),
/tmp/ipykernel_3747/3758758826.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'attention_mask': torch.tensor(self.masks[idx], dtype=torch.long),


              precision    recall  f1-score   support

       anger       0.74      0.75      0.75       110
        fear       0.89      0.65      0.75        65
       happy       0.68      0.70      0.69       101
        love       0.68      0.80      0.73        64
     sadness       0.59      0.60      0.59       100

    accuracy                           0.70       440
   macro avg       0.72      0.70      0.70       440
weighted avg       0.71      0.70      0.70       440



In [ ]:
y_pred_bert = np.array(y_pred_bert)
np.save('y_pred_bert.npy', y_pred_bert)

## **B. IndoBERTweet (Shaw dkk.)**

### **1) Tokenization**

In [ ]:
model_checkpoint_tw = "indolem/indobertweet-base-uncased"
tokenizer_tw = AutoTokenizer.from_pretrained(model_checkpoint_tw)

def prep_indobertweet(text):
    text = str(text).lower()
    # IndoBERTweet Preprocessing: @USER, HTTPURL
    text = re.sub(r'\[username\]', '@USER', text)
    text = re.sub(r'\[url\]', 'HTTPURL', text)
    return text.strip()

X_train_tw_tweet = df_train['tweet'].apply(prep_indobertweet).tolist()
X_valid_tw_tweet = df_valid['tweet'].apply(prep_indobertweet).tolist()
X_test_tw_tweet  = df_test['tweet'].apply(prep_indobertweet).tolist()

def encode_tweet(texts, max_len=43):
    encoded = tokenizer_tw(
        texts,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    return encoded['input_ids'], encoded['attention_mask']

train_ids_tw, train_mask_tw = encode_tweet(X_train_tw_tweet)
valid_ids_tw, valid_mask_tw = encode_tweet(X_valid_tw_tweet)
test_ids_tw, test_mask_tw   = encode_tweet(X_test_tw_tweet)

train_dataset_tw = EmotDataset(train_ids_tw, train_mask_tw, y_train)
valid_dataset_tw = EmotDataset(valid_ids_tw, valid_mask_tw, y_valid)
test_dataset_tw  = EmotDataset(test_ids_tw, test_mask_tw, y_test)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

### **2) Custom Head**

In [ ]:
import torch.nn as nn
from transformers import AutoModel

class IndoBERTweetCustom(nn.Module):
    def __init__(self, num_labels):
        super(IndoBERTweetCustom, self).__init__()
        self.num_labels = num_labels
        self.bert = AutoModel.from_pretrained("indolem/indobertweet-base-uncased")

        # (Shaw et al.)
        self.dense1 = nn.Linear(self.bert.config.hidden_size, 64)
        self.dropout = nn.Dropout(0.05)
        self.dense2 = nn.Linear(64, 16)
        self.classifier = nn.Linear(16, num_labels)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # GLOBAL AVERAGE POOLING
        mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
        sum_embeddings = torch.sum(outputs.last_hidden_state * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask

        # CUSTOM CLASSIFICATION HEAD
        x = self.dense1(pooled_output)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.dense2(x)
        x = torch.relu(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            return {"loss": loss, "logits": logits}

        return {"logits": logits}

model_tweet = IndoBERTweetCustom(
    num_labels=len(class_mapping),
).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### **3) Training**

In [ ]:
# IndoBERTweet Parameter (Shaw et al.)
training_args_tw = TrainingArguments(
    output_dir='./indobertweet_results',
    num_train_epochs=25,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    weight_decay=0.005,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer_tw = Trainer(
    model=model_tweet,
    args=training_args_tw,
    train_dataset=train_dataset_tw,
    eval_dataset=valid_dataset_tw,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

trainer_tw.train()

print("\n[Evaluation IndoBERTweet on TEST Data]")
predictions_tw = trainer_tw.predict(test_dataset_tw)
y_pred_tweet = np.argmax(predictions_tw.predictions, axis=1)
print(classification_report(y_test, y_pred_tweet, target_names=label_encoder.classes_))

/tmp/ipykernel_527/376574302.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(self.ids[idx], dtype=torch.long),
/tmp/ipykernel_527/376574302.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'attention_mask': torch.tensor(self.masks[idx], dtype=torch.long),


Epoch,Training Loss,Validation Loss
1,0.301774,1.156352
2,0.240738,1.275535
3,0.195624,1.214498
4,0.159867,1.266547
5,0.139929,1.353974
6,0.112259,1.365884


/tmp/ipykernel_527/376574302.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(self.ids[idx], dtype=torch.long),
/tmp/ipykernel_527/376574302.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'attention_mask': torch.tensor(self.masks[idx], dtype=torch.long),
/tmp/ipykernel_527/376574302.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(self.ids[idx], dtype=torch.long),
/tmp/ipykernel_527/376574302.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sour


[Evaluation IndoBERTweet on TEST Data]


/tmp/ipykernel_527/376574302.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(self.ids[idx], dtype=torch.long),
/tmp/ipykernel_527/376574302.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'attention_mask': torch.tensor(self.masks[idx], dtype=torch.long),


              precision    recall  f1-score   support

       anger       0.82      0.79      0.81       110
        fear       0.75      0.82      0.78        65
       happy       0.79      0.69      0.74       101
        love       0.74      0.86      0.80        64
     sadness       0.68      0.68      0.68       100

    accuracy                           0.76       440
   macro avg       0.76      0.77      0.76       440
weighted avg       0.76      0.76      0.76       440



### **4) Save Model**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
model_final_path_drive = "/content/drive/MyDrive/indobertweet-emotion-v1"

trainer_tw.save_model(model_final_path_drive)
tokenizer_tw.save_pretrained(model_final_path_drive)

('/content/drive/MyDrive/indobertweet-emotion-v1/tokenizer_config.json',
 '/content/drive/MyDrive/indobertweet-emotion-v1/tokenizer.json')

### **5) Load Model to Save y_pred_tweet**

In [ ]:
from safetensors.torch import load_file

model_tw = IndoBERTweetCustom(
    num_labels=len(class_mapping),
)

model_path = "/content/drive/MyDrive/indobertweet-emotion-v1/model.safetensors"
weight = load_file(model_path)
model_tw.load_state_dict(weight)
model_tw.to(device)
model_tw.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


IndoBERTweetCustom(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31923, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

In [ ]:
y_pred_tweet = np.array(y_pred_tweet)
np.save('y_pred_tweet.npy', y_pred_tweet)

In [ ]:
y_test = np.array(y_test)
np.save('y_test.npy', y_test)